In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
print(f'IN_COLAB={IN_COLAB}')


### Environment detection
We detect Colab vs local execution up front so paths and installation behavior stay deterministic.

### Configure workspace paths
On Colab we mount Drive for persistent checkpoints; locally we use the repository root.

In [ ]:
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/piano_model')
    BASE_PATH.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = BASE_PATH / 'piano_midi_model'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
    BASE_PATH = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT={PROJECT_ROOT}')


### Install dependencies
This installs the Colab CUDA requirements in GPU runs and CPU-safe requirements locally.

In [ ]:
import subprocess
import sys

req_file = PROJECT_ROOT / ('requirements_colab.txt' if IN_COLAB else 'requirements.txt')
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=False)
else:
    print(f'Requirements file not found: {req_file}')


### Load configs and instantiate hybrid model (Mamba+CfC)
This is the main experiment configuration where both Mamba and CfC dynamics are enabled.

In [ ]:
from config import DataConfig, ModelConfig, TrainConfig
from model.hybrid import PianoHybridModel

data_cfg = DataConfig()
model_cfg = ModelConfig(use_cfc=True, use_mamba=True)
train_cfg = TrainConfig(max_epochs=50, batch_size=8, grad_accumulation_steps=4, device='auto')

if IN_COLAB:
    data_cfg.maestro_path = str(BASE_PATH / 'maestro-v3.0.0')
    data_cfg.tokenizer_path = str(BASE_PATH / 'tokenizer.json')
    data_cfg.processed_path = str(BASE_PATH / 'processed')
    train_cfg.checkpoint_dir = str(BASE_PATH / 'checkpoints_hybrid')

model_cfg.vocab_size = data_cfg.vocab_size
model_cfg.max_sequence_length = data_cfg.max_sequence_length
hybrid_model = PianoHybridModel(model_cfg)
hybrid_model


### Log model summary
This captures parameter breakdown and active backend details for experiment traceability.

In [ ]:
from utils.logging_utils import log_model_summary

log_model_summary(hybrid_model, model_cfg)


### Create DataLoaders
We keep the exact same data split policy as other runs for fair ablation comparisons.

In [ ]:
from data.dataset import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(data_cfg, train_cfg)
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))


### Initialize trainer
The trainer handles optimization, scheduler stepping, periodic checkpointing, and best-model selection by validation loss.

In [ ]:
from data.tokenizer import PianoTokenizer
from training.trainer import Trainer

tokenizer = PianoTokenizer.load(data_cfg.tokenizer_path)
trainer = Trainer(
    model=hybrid_model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    data_config=data_cfg,
    tokenizer=tokenizer,
)
trainer


### Train hybrid model
This is the primary run for testing whether CfC improves temporal continuation quality over Mamba-only and baseline models.

In [ ]:
history_hybrid = trainer.train()
history_hybrid.keys()


### Plot side-by-side loss curves for all models
This overlays baseline, Mamba-only, and hybrid curves from saved checkpoint histories for direct convergence comparison.

In [ ]:
import matplotlib.pyplot as plt
import torch

history_map = {'hybrid': history_hybrid}
candidate_states = {
    'baseline': [PROJECT_ROOT / 'checkpoints_baseline' / 'best_state.pt', BASE_PATH / 'checkpoints_baseline' / 'best_state.pt'],
    'mamba_only': [PROJECT_ROOT / 'checkpoints_mamba_only' / 'best_state.pt', BASE_PATH / 'checkpoints_mamba_only' / 'best_state.pt'],
}

for name, paths in candidate_states.items():
    for p in paths:
        if p.exists():
            history_map[name] = torch.load(p, map_location='cpu').get('history', {})
            print(f'Loaded {name} history from {p}')
            break

plt.figure(figsize=(10, 4))
for name, hist in history_map.items():
    vals = hist.get('val_loss', [])
    if len(vals) > 0:
        plt.plot(vals, label=f'{name}_val', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Validation Curves: Baseline vs Mamba-only vs Hybrid')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Quantitative evaluation on test seeds
We run the metric suite to summarize key consistency, density consistency, and rhythmic behavior of generated continuations.

In [ ]:
from evaluation.metrics import evaluate_dataset

summary_metrics = evaluate_dataset(hybrid_model, tokenizer, test_loader, data_cfg)
for key, stats in summary_metrics.items():
    print(f'{key}: mean={stats["mean"]:.4f}, std={stats["std"]:.4f}')


### Generate one continuation, visualize, and listen
This qualitative sample complements quantitative metrics and helps inspect phrasing/rhythm continuity.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import Audio, display

from utils.midi_utils import compare_pianorolls

seed_batch, _ = next(iter(test_loader))
seed_tokens = seed_batch[0].tolist()
generated = hybrid_model.generate(seed_tokens, max_new_tokens=data_cfg.continuation_length)
generated_continuation = generated[len(seed_tokens):]

out_dir = Path(train_cfg.checkpoint_dir) / 'qualitative'
out_dir.mkdir(parents=True, exist_ok=True)
seed_mid = out_dir / 'hybrid_seed.mid'
gen_full_mid = out_dir / 'hybrid_generated_full.mid'
gen_cont_mid = out_dir / 'hybrid_generated_continuation.mid'
tokenizer.decode(seed_tokens, seed_mid)
tokenizer.decode(generated, gen_full_mid)
tokenizer.decode(generated_continuation, gen_cont_mid)
compare_pianorolls(seed_mid, gen_cont_mid)

wav_path = out_dir / 'hybrid_generated.wav'
rendered = False
soundfont_candidates = ['/usr/share/sounds/sf2/FluidR3_GM.sf2', '/usr/share/soundfonts/default.sf2']
soundfont = next((p for p in soundfont_candidates if Path(p).exists()), None)
if IN_COLAB and shutil.which('fluidsynth') is None:
    subprocess.run(['apt-get', '-y', 'install', 'fluidsynth'], check=False)
if shutil.which('fluidsynth') and soundfont:
    cmd = ['fluidsynth', '-ni', soundfont, str(gen_full_mid), '-F', str(wav_path), '-r', '44100']
    subprocess.run(cmd, check=False)
    rendered = wav_path.exists()
if rendered:
    display(Audio(filename=str(wav_path), rate=44100))
else:
    print('Audio rendering skipped (fluidsynth/soundfont missing).')


### Save explicit final checkpoint
This creates a tagged final checkpoint in addition to automatic best/latest snapshots.

In [ ]:
final_val = history_hybrid['val_loss'][-1] if history_hybrid['val_loss'] else 0.0
trainer.save_checkpoint(epoch=train_cfg.max_epochs, val_loss=final_val, tag='hybrid_final')
print(f'Checkpoint directory: {train_cfg.checkpoint_dir}')
